# Google Play Store — Data Cleaning Pipeline (FINAL)
### Section A | Group 13

This notebook transforms the raw dataset into a **zero-null, analysis-ready dataset** using Mean / Median / Mode.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/Google-Playstore-raw.csv'
CLEAN_PATH = '../data/Google-Playstore-Cleaned.csv'

df = pd.read_csv(RAW_PATH)

print(f'Loaded: {len(df):,} rows, {len(df.columns)} columns')
print(f'Total nulls: {df.isnull().sum().sum():,}')

---
## Step 1: Drop rows with missing App Name

In [ ]:
df.dropna(subset=['App Name'], inplace=True)
print(f'Remaining rows: {len(df):,}')

---
## Step 2: Handle Rating & Rating Count

In [ ]:
# Rating → Mean
rating_mean = df['Rating'].mean()
df['Rating'] = df['Rating'].fillna(rating_mean)

# Rating Count → Median (skewed)
rating_count_median = df['Rating Count'].median()
df['Rating Count'] = df['Rating Count'].fillna(rating_count_median)

print('Rating mean used:', rating_mean)
print('Rating Count median used:', rating_count_median)

---
## Step 3: Convert Size to Numeric

In [ ]:
def convert_size(size_str):
    if pd.isna(size_str) or 'Varies with device' in str(size_str):
        return np.nan
    size_str = str(size_str).upper().replace(',', '')
    if 'M' in size_str:
        return float(size_str.replace('M', ''))
    elif 'K' in size_str:
        return float(size_str.replace('K', '')) / 1024
    elif 'G' in size_str:
        return float(size_str.replace('G', '')) * 1024
    return np.nan

df['Size_MB'] = df['Size'].apply(convert_size)

size_median = df['Size_MB'].median()
df['Size_MB'] = df['Size_MB'].fillna(size_median)
df['Size'] = df['Size'].fillna('Varies with device')

print('Size median used:', size_median)

---
## Step 4: Convert Installs

In [ ]:
installs_median = df['Minimum Installs'].median()

df['Installs_Numeric'] = (
    df['Installs'].str.replace('+', '', regex=False)
                  .str.replace(',', '', regex=False)
                  .apply(pd.to_numeric, errors='coerce')
                  .fillna(installs_median)
)

df['Minimum Installs'] = df['Minimum Installs'].fillna(installs_median)
df['Installs'] = df['Installs'].fillna(f'{int(installs_median)}+')

print('Installs median used:', installs_median)

---
## Step 5: Currency (Mode)

In [ ]:
currency_mode = df['Currency'].mode().iloc[0]
df['Currency'] = df['Currency'].fillna(currency_mode)
df['Currency'] = df['Currency'].replace('XXX', currency_mode)

print('Currency mode used:', currency_mode)

---
## Step 6: Other Categorical Columns

In [ ]:
df['Minimum Android'] = df['Minimum Android'].fillna(df['Minimum Android'].mode().iloc[0])
df['Developer Id'] = df['Developer Id'].fillna(df['Developer Id'].mode().iloc[0])
df['Developer Email'] = df['Developer Email'].fillna(df['Developer Email'].mode().iloc[0])
df['Developer Website'] = df['Developer Website'].fillna(df['Developer Website'].mode().iloc[0])
df['Released'] = df['Released'].fillna(df['Released'].mode().iloc[0])
df['Privacy Policy'] = df['Privacy Policy'].fillna(df['Privacy Policy'].mode().iloc[0])

---
## Step 7: Convert Dates

In [ ]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

---
## Step 8: Final Check

In [ ]:
if df.isnull().sum().sum() == 0:
    print('✅ ZERO NULLS — CLEAN DATASET')
else:
    print('⚠️ Nulls still exist')

print('Final shape:', df.shape)

---
## Step 9: Save Dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
print('Saved successfully')